# AI 기본법 Advanced RAG 및 평가

기존 Qdrant 컬렉션을 사용해 다음 흐름을 실행합니다.

```text
기존 조·항 청크 → Dense 기준선 → Hybrid(Dense+BM25+RRF)
→ Qwen Reranking → 근거 기반 답변 → Retrieval/Answer 평가
```

프로젝트의 `common.ai_model`, `common.qdrant`를 재사용합니다. 문서 청킹·임베딩·저장은 다시 수행하지 않습니다.

필요 패키지:

```bash
uv add rank-bm25 sentence-transformers "transformers>=4.51.0"
```


## 1. 공통 설정과 기존 Qdrant 컬렉션 연결


In [20]:
from pathlib import Path
import sys
import re
import time
from collections import Counter

import pandas as pd
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from qdrant_client.models import FieldCondition, Filter, MatchValue

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "common" / "ai_model.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

from common.ai_model import get_embedding_model, get_llm_model
from common.qdrant import get_qdrant_client

embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()

COLLECTION_NAME = "ai_basic_law_adaptive_anchor_v1"

TOP_K = 5
CANDIDATE_K = 20

available = [item.name for item in qdrant_client.get_collections().collections]
print("사용 가능한 컬렉션:", available)
if COLLECTION_NAME not in available:
    raise ValueError(f"컬렉션이 없습니다: {COLLECTION_NAME}")


사용 가능한 컬렉션: ['ai_basic_law_adaptive_anchor_v1', 'ai_basic_law_article_paragraph_item_v1', 'ai_basic_law_article_paragraph_v1', 'ai_basic_law_articles_v1', 'ai_basic_law_hierarchy_v1', 'ai_basic_law_xml', 'law_articles']


## 2. 기존 청크 불러오기

Qdrant의 모든 payload를 읽어 BM25 인덱스를 구성합니다. Qdrant 포인트 ID를 공통 `chunk_id`로 사용합니다.


In [21]:
all_chunks = []
offset = None

while True:
    records, offset = qdrant_client.scroll(
        collection_name=COLLECTION_NAME,
        limit=256,
        offset=offset,
        with_payload=True,
        with_vectors=False,
    )
    for point in records:
        payload = dict(point.payload or {})
        content = payload.get("content")
        if not isinstance(content, str) or not content.strip():
            continue
        payload["chunk_id"] = str(point.id)
        all_chunks.append(payload)

    if offset is None:
        break

if not all_chunks:
    raise ValueError("Qdrant에서 content가 있는 청크를 찾지 못했습니다.")

# Dense와 BM25에는 parent를 제외한 child/anchor만 사용합니다.
chunks = [chunk for chunk in all_chunks if chunk.get("search_enabled") is True]
chunks_by_key = {
    chunk.get("chunk_key"): chunk
    for chunk in all_chunks
    if chunk.get("chunk_key")
}

if not chunks:
    raise ValueError("search_enabled=True인 검색 청크가 없습니다.")

print("전체 저장 청크:", len(all_chunks))
print("검색용 청크:", len(chunks))
print("청크 역할:", dict(Counter(c.get("chunk_role", "unknown") for c in all_chunks)))
print("검색 단위:", dict(Counter(c.get("unit_level", "unknown") for c in chunks)))
print("제2조 검색 청크:", sum(c.get("article") == "제2조" for c in chunks))


전체 저장 청크: 118
검색용 청크: 111
청크 역할: {'child': 110, 'parent': 7, 'anchor': 1}
검색 단위: {'subitem': 11, 'article': 39, 'item': 12, 'paragraph': 44, 'addendum': 3, 'reason': 2}
제2조 검색 청크: 23


## 3. Dense 기준선


In [22]:
import asyncio
from openai import RateLimitError

query_embedding_cache = {}

async def cache_query_embeddings(questions):
    missing_questions = list(dict.fromkeys(
        question
        for question in questions
        if question not in query_embedding_cache
    ))

    if missing_questions:
        attempt = 0
        while True:
            attempt += 1
            try:
                # 40개 질문도 가능한 한 번의 배치 요청으로 처리합니다.
                vectors = await embedding_client.aembed_documents(missing_questions)
                query_embedding_cache.update(zip(missing_questions, vectors))
                print(f"질문 임베딩 캐시 완료: {len(query_embedding_cache)}개")
                break
            except Exception as error:
                # 라이브러리에 따라 RateLimitError 이름이 달라도 429만 재시도합니다.
                message = str(error).lower()
                is_rate_limit = (
                    isinstance(error, RateLimitError)
                    or '429' in message
                    or 'rate limit' in message
                    or 'rate_limit' in message
                )
                if not is_rate_limit:
                    raise

                wait_seconds = 65
                print(
                    f"Rate Limit 발생: {wait_seconds}초 후 무한 재시도 "
                    f"(현재 {attempt}회 실패)"
                )
                await asyncio.sleep(wait_seconds)

    return [query_embedding_cache[question] for question in questions]

async def dense_search(question, k=TOP_K):
    if question not in query_embedding_cache:
        await cache_query_embeddings([question])

    query_vector = query_embedding_cache[question]
    result = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(must=[
            FieldCondition(
                key="search_enabled",
                match=MatchValue(value=True),
            )
        ]),
        limit=k,
    )
    return [
        {
            **dict(point.payload or {}),
            "chunk_id": str(point.id),
            "dense_score": float(point.score),
        }
        for point in result.points
        if point.payload and point.payload.get("content")
    ]


## 4. C: Hybrid Search

Dense와 BM25 결과를 RRF로 결합합니다. 조사와 질문형 어미 일부를 정규화해 `인공지능이란`과 `인공지능` 같은 표현 차이를 줄입니다.


In [23]:
def tokenize(text):
    raw_tokens = re.findall(r"[가-힣]+|[a-z]+|\d+", text.lower())
    suffixes = ("인가요", "이란", "이랑", "에서", "으로", "에게", "은", "는", "이", "가", "을", "를", "의")
    normalized = []
    for token in raw_tokens:
        for suffix in suffixes:
            if len(token) > len(suffix) + 1 and token.endswith(suffix):
                token = token[:-len(suffix)]
                break
        if token:
            normalized.append(token)
    return normalized

def searchable_text(chunk):
    fields = [
        "law_name", "chapter", "section", "article",
        "paragraph", "item", "subitem", "title", "content",
    ]
    return " ".join(
        str(chunk.get(field) or "")
        for field in fields
    )

bm25 = BM25Okapi([tokenize(searchable_text(chunk)) for chunk in chunks])

def bm25_search(question, k=CANDIDATE_K):
    scores = bm25.get_scores(tokenize(question))
    ranked_indices = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
    return [
        {**chunks[index], "bm25_score": float(scores[index])}
        for index in ranked_indices
        if scores[index] > 0
    ][:k]

def rrf(ranked_lists, k=60):
    scores = {}
    documents = {}
    for ranked in ranked_lists:
        for rank, document in enumerate(ranked, start=1):
            chunk_id = document["chunk_id"]
            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1 / (k + rank)
            documents[chunk_id] = document

    ranked_ids = sorted(scores, key=scores.get, reverse=True)
    return [
        {**documents[chunk_id], "rrf_score": scores[chunk_id]}
        for chunk_id in ranked_ids
    ]

async def hybrid_search(question, k=TOP_K, candidate_k=CANDIDATE_K):
    dense_results = await dense_search(question, candidate_k)
    keyword_results = bm25_search(question, candidate_k)
    return rrf([dense_results, keyword_results])[:k]


## 5. D: Qwen Reranking

Hybrid 후보 안에서 질문과 직접 관련된 근거가 위로 오도록 재정렬합니다. 후보에 없는 근거를 새로 찾는 단계는 아닙니다.


In [24]:
import torch
from sentence_transformers import CrossEncoder

QWEN_RERANKER_ID = "Qwen/Qwen3-Reranker-0.6B"
qwen_reranker = None

def load_reranker():
    global qwen_reranker
    if qwen_reranker is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Qwen Reranker 로딩:", device)
        qwen_reranker = CrossEncoder(
            QWEN_RERANKER_ID,
            device=device,
            max_length=2048,
        )
    return qwen_reranker

def rerank(question, candidates, k=TOP_K):
    if not candidates:
        return []

    model = load_reranker()
    pairs = [
        (
            question,
            f"{doc.get('article', '')} {doc.get('paragraph', '')} "
            f"{doc.get('title', '')}\n{doc['content']}",
        )
        for doc in candidates
    ]
    scores = model.predict(
        pairs,
        batch_size=1,
        activation_fn=torch.nn.Sigmoid(),
        show_progress_bar=True,
    )
    scored = [
        {**doc, "rerank_score": float(score)}
        for doc, score in zip(candidates, scores)
    ]
    return sorted(scored, key=lambda doc: doc["rerank_score"], reverse=True)[:k]

async def hybrid_rerank_search(question, k=TOP_K, candidate_k=CANDIDATE_K):
    hybrid_candidates = await hybrid_search(
        question,
        k=candidate_k,
        candidate_k=candidate_k,
    )
    return rerank(question, hybrid_candidates, k=k)


## 6. 단일 질문으로 전후 비교

`USE_RERANK=False`로 바꾸면 Qwen 로딩 없이 Dense와 Hybrid만 비교할 수 있습니다.


In [25]:
query = "고영향 인공지능이란 무엇인가요?"
USE_RERANK = False

started = time.perf_counter()
dense_results = await dense_search(query, TOP_K)
hybrid_results = await hybrid_search(query, TOP_K)
reranked_results = (
    await hybrid_rerank_search(query, TOP_K)
    if USE_RERANK
    else []
)

methods = {
    "Dense": dense_results,
    "Hybrid": hybrid_results,
}
if USE_RERANK:
    methods["Hybrid+Reranking"] = reranked_results

for method, results in methods.items():
    print(f"\n[{method}]")
    for rank, doc in enumerate(results, start=1):
        score = doc.get("rerank_score", doc.get("rrf_score", doc.get("dense_score")))
        print(
            rank,
            f"{doc.get('chunk_role', '')}/{doc.get('unit_level', '')}",
            doc.get("article", ""),
            doc.get("paragraph", ""),
            doc.get("item", ""),
            doc.get("subitem", ""),
            doc.get("title", ""),
            "score=", round(score, 4) if score is not None else None,
        )
        print(doc["content"][:600])

print("\n실행 시간(초):", round(time.perf_counter() - started, 2))


질문 임베딩 캐시 완료: 1개

[Dense]
1 child/article 제33조    고영향 인공지능의 확인 score= 0.5928
제33조(고영향 인공지능의 확인)
① 인공지능사업자는 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 그 인공지능이 고영향 인공지능에 해당하는지에 대하여 사전에 검토하여야 하며, 필요한 경우 과학기술정보통신부장관에게 고영향 인공지능에 해당하는지 여부의 확인을 요청할 수 있다.
② 과학기술정보통신부장관은 제1항에 따른 요청이 있는 경우 고영향 인공지능 해당 여부를 확인하여야 하며, 필요한 경우 전문위원회를 설치하여 관련 자문을 받을 수 있다.
③ 과학기술정보통신부장관은 고영향 인공지능의 기준과 예시 등에 관한 가이드라인을 수립하여 보급할 수 있다.
④ 그 밖에 제1항에 따른 확인 절차 등에 관하여 필요한 사항은 대통령령으로 정한다.
2 anchor/item 제2조  4호  정의 score= 0.5922
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.
3 child/subitem 제2조  4호 바목 정의 score= 0.5837
바. 범죄 수사나 체포 업무를 위한 생체인식정보(얼굴ㆍ지문ㆍ홍채 및 손바닥 정맥 등 개인을 식별할 수 있는 신체적ㆍ생리적ㆍ행동적 특징에 관한 개인정보를 말한다)의 분석ㆍ활용
4 child/subitem 제2조  4호 사목 정의 score= 0.5816
사. 채용, 대출 심사 등 개인의 권리ㆍ의무 관계에 중대한 영향을 미치는 판단 또는 평가
5 child/subitem 제2조  4호 자목 정의 score= 0.5794
자. 공공서비스 제공에 필요한 자격 확인 및 결정 또는 비용징수 등 국민에게 영향을 미치는 국가, 지방자치단체, 「공공기관의 운영에 관한 법률」 제4조에 따른 공공기관 등(이하 "국가기관등"이라 한다)의 의사결정

[Hybrid]
1 ch

## 7. Golden Test Set

실제 평가에서는 `eval/golden_set.jsonl`로 분리할 수 있습니다. 여기서는 노트북 실행 예시를 위해 대표 질문을 정의합니다.


In [26]:
representative_golden_set = [
    {
        "id": "q001",
        "question": "이 법의 목적은 무엇인가요?",
        "gold_articles": ["제1조"],
        "reference_points": ["국민의 권익과 존엄성 보호", "삶의 질 향상", "국가경쟁력 강화"],
        "answerable": True,
    },
    {
        "id": "q002",
        "question": "고영향 인공지능이란 무엇인가요?",
        "gold_articles": ["제2조"],
        "reference_points": ["생명", "신체의 안전", "기본권", "중대한 영향"],
        "answerable": True,
    },
    {
        "id": "q003",
        "question": "고영향 인공지능 해당 여부는 어떻게 확인하나요?",
        "gold_articles": ["제33조"],
        "reference_points": ["사전 검토", "과학기술정보통신부장관", "확인 요청"],
        "answerable": True,
    },
    {
        "id": "q004",
        "question": "고영향 인공지능의 확인 절차와 영향평가 의무는 무엇인가요?",
        "gold_articles": ["제33조", "제35조"],
        "reference_points": ["확인", "영향평가"],
        "answerable": True,
    },
    {
        "id": "q005",
        "question": "이 법은 국외에서 이루어진 행위에도 적용되나요?",
        "gold_articles": ["제4조"],
        "reference_points": ["국내 시장", "이용자", "영향"],
        "answerable": True,
    },
]

# 실제 평가는 golden_set_v2.jsonl 전체 문항을 사용합니다.
import json

GOLDEN_SET_PATH = PROJECT_ROOT / "eval" / "golden_set_v2.jsonl"
with GOLDEN_SET_PATH.open(encoding="utf-8") as file:
    golden_set = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print(f"Golden Set 경로: {GOLDEN_SET_PATH}")
print(f"Golden Set 문항 수: {len(golden_set)}")
display(pd.DataFrame(golden_set).head())


Golden Set 경로: /home/aiuser/rag_mini_pjt_template/eval/golden_set_v2.jsonl
Golden Set 문항 수: 45


,id,type,question,answerable,answer_articles,related_articles,answer_points,added_in
0,q01,정의,고영향 인공지능이란 무엇인가요?,True,[제2조],[제2조],[사람의 생명·신체 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 ...,NaN
1,q02,정의,생성형 인공지능은 어떻게 정의되나요?,True,[제2조],[제2조],"[입력 데이터의 구조와 특성을 모방하여 글, 소리, 그림, 영상 등을 생성하는 인공...",NaN
2,q03,정의,인공지능사업자는 누구를 말하나요?,True,[제2조],[제2조],"[인공지능산업과 관련된 사업을 하는 법인·단체·개인·국가기관등, 인공지능개발사업자와...",NaN
3,q04,정의,인공지능산업이란 무엇인가요?,True,[제2조],[제2조],[인공지능 또는 인공지능기술을 활용한 제품을 개발·제조·유통하거나 관련 서비스를 제...,NaN
4,q05,정의,인공지능취약계층이란 어떤 사람들인가요?,True,[제3조],[제3조],[장애인·고령자 등 대통령령으로 정하는 취약계층],NaN


In [27]:
# import json
# import pandas as pd

# GOLDEN_SET_PATH = PROJECT_ROOT / "eval" / "golden_set_v2.jsonl"

# with GOLDEN_SET_PATH.open(encoding="utf-8") as file:
#     golden_set = [
#         json.loads(line)
#         for line in file
#         if line.strip()
#     ]

# print(f"Golden Set 문항 수: {len(golden_set)}")
# pd.DataFrame(golden_set).head()

## 8. Retrieval 평가: Hit@5, Recall@5, MRR@5

조문 전체와 항 청크가 함께 검색돼도 같은 조문은 하나의 근거로 계산합니다.


In [28]:
def unique_articles(results):
    return list(dict.fromkeys(
        doc.get("article", "")
        for doc in results
        if doc.get("article")
    ))

def retrieval_metrics(results, gold_articles, k=TOP_K):
    retrieved = unique_articles(results[:k])
    gold = set(gold_articles)
    found = gold.intersection(retrieved)

    # 관련 조문도 없는 범위 밖 질문은 Retrieval 평균에서 제외합니다.
    if not gold:
        return {
            f"Hit@{k}": None,
            f"Recall@{k}": None,
            f"MRR@{k}": None,
            "retrieved_articles": retrieved,
        }

    hit = 1.0 if found else 0.0
    recall = len(found) / len(gold)

    reciprocal_rank = 0.0
    for rank, article in enumerate(retrieved, start=1):
        if article in gold:
            reciprocal_rank = 1.0 / rank
            break

    return {
        f"Hit@{k}": hit,
        f"Recall@{k}": recall,
        f"MRR@{k}": reciprocal_rank,
        "retrieved_articles": retrieved,
    }

async def evaluate_retrieval(use_reranking=True):
    rows = []

    # Golden Set 전체를 한 번의 API 요청으로 임베딩하고 캐시에 저장합니다.
    await cache_query_embeddings([
        item["question"]
        for item in golden_set
    ])

    for index, item in enumerate(golden_set, start=1):
        question = item["question"]

        search_methods = {
            "Dense": await dense_search(question, TOP_K),
            "Hybrid": await hybrid_search(question, TOP_K),
        }
        if use_reranking:
            search_methods["Hybrid+Reranking"] = await hybrid_rerank_search(
                question,
                TOP_K,
            )

        gold_articles = item.get("gold_articles")
        if gold_articles is None:
            gold_articles = (
                item.get("answer_articles", [])
                if item.get("answerable", True)
                else item.get("related_articles", [])
            )

        for method, results in search_methods.items():
            metrics = retrieval_metrics(
                results,
                gold_articles,
                TOP_K,
            )
            rows.append({
                "question_id": item["id"],
                "method": method,
                **metrics,
            })

        print(f"[{index}/{len(golden_set)}] {item['id']} 완료")

    detail = pd.DataFrame(rows)
    metric_columns = [f"Hit@{TOP_K}", f"Recall@{TOP_K}", f"MRR@{TOP_K}"]
    summary = detail.groupby("method")[metric_columns].mean().reset_index()
    return detail, summary

# Qwen 평가를 생략하려면 False로 변경
retrieval_detail, retrieval_summary = await evaluate_retrieval(
    use_reranking=True,
)
display(retrieval_summary)
display(retrieval_detail)


질문 임베딩 캐시 완료: 45개
Qwen Reranker 로딩: cuda


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

[1/45] q01 완료


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

...

[44/45] q44 완료


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

[45/45] q45 완료


,method,Hit@5,Recall@5,MRR@5
0,Dense,0.809524,0.773810,0.670238
1,Hybrid,0.928571,0.892857,0.721032
2,Hybrid+Reranking,0.904762,0.880952,0.845238


,question_id,method,Hit@5,Recall@5,MRR@5,retrieved_articles
0,q01,Dense,1.0,1.0,0.5,"[제33조, 제2조]"
1,q01,Hybrid,1.0,1.0,0.5,"[제33조, 제2조, 제35조, 제34조]"
2,q01,Hybrid+Reranking,1.0,1.0,1.0,"[제2조, 제35조, 제33조, 제34조]"
3,q02,Dense,1.0,1.0,1.0,[제2조]
4,q02,Hybrid,1.0,1.0,1.0,"[제2조, 제31조]"
...,...,...,...,...,...,...
130,q44,Hybrid,1.0,1.0,1.0,[제6조]
131,q44,Hybrid+Reranking,1.0,1.0,0.5,"[제37조, 제6조, 제17조의2, 제22조의2, 제8조]"
132,q45,Dense,0.0,0.0,0.0,"[제32조, 제30조, 제24조, 제33조, 제31조]"
133,q45,Hybrid,0.0,0.0,0.0,"[제33조, 제29조, 제30조, 제32조, 제2조]"


## 9. 근거 기반 답변 생성

검색된 항 청크에 대응하는 조문 전체 청크가 있으면 조문 전체를 Context로 사용합니다.


In [29]:
def build_context(results):
    selected = []
    seen = set()

    for result in results:
        # 긴 조문은 parent_id로 전체 조문을 사용하고,
        # 짧은 조문은 parent_id가 자기 자신의 chunk_key를 가리킵니다.
        parent_id = result.get("parent_id")
        document = chunks_by_key.get(parent_id, result)
        key = document.get("chunk_id", document.get("chunk_key"))
        if key in seen:
            continue
        seen.add(key)
        selected.append(document)

    context_parts = []
    sources = []

    for index, doc in enumerate(selected, start=1):
        source_id = f"S{index}"
        context_parts.append(
            f"[{source_id}] {doc.get('article', '')} {doc.get('title', '')}\n"
            f"{doc['content']}"
        )
        sources.append({
            "source_id": source_id,
            "article": doc.get("article", ""),
            "content": doc["content"],
        })

    return "\n\n".join(context_parts), sources

llm = get_llm_model(max_tokens=1200)
answer_cache = {}

async def answer_question(question, results):
    context, sources = build_context(results)
    cache_key = (
        question,
        tuple(source.get("article", "") for source in sources),
        context,
    )
    if cache_key in answer_cache:
        return answer_cache[cache_key]

    messages = [
        (
            "system",
            "제공된 법령 근거만 사용해 한국어로 답하세요. "
            "각 핵심 주장 뒤에 [S1] 형태로 출처를 표시하세요. "
            "근거가 부족하면 확인할 수 없다고 답하고 추측하지 마세요.",
        ),
        (
            "human",
            f"질문: {question}\n\n법령 근거:\n{context}",
        ),
    ]

    attempt = 0
    while True:
        attempt += 1
        try:
            response = await llm.ainvoke(messages)
            result = {
                "answer": response.content,
                "sources": sources,
            }
            answer_cache[cache_key] = result
            return result
        except Exception as error:
            error_text = str(error).lower()
            is_rate_limit = "429" in error_text or "rate limit" in error_text
            if not is_rate_limit:
                raise
            wait_seconds = 65
            print(
                f"LLM Rate Limit 발생: {wait_seconds}초 후 무한 재시도 "
                f"(현재 {attempt}회 실패)"
            )
            await asyncio.sleep(wait_seconds)

# 예시: 최종 검색 결과로 답변 생성
final_results = (
    reranked_results
    if USE_RERANK
    else hybrid_results
)
answer_result = await answer_question(query, final_results)
print(answer_result["answer"])
print("\nSources:")
for source in answer_result["sources"]:
    print(source["source_id"], source["article"])


고영향 인공지능이란, 사람의 생명·신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템을 말합니다. [S2]

또한 이러한 인공지능은 다음과 같은 영역에서 활용되는 것을 의미합니다: 에너지 공급, 먹는물 생산 공정, 보건의료 제공 및 이용체계, 의료기기 및 디지털의료기기 개발·이용, 원자력시설의 안전한 관리·운영, 범죄 수사나 체포 업무를 위한 생체인식정보의 분석·활용, 채용·대출 심사 등 권리·의무에 중대한 영향을 미치는 판단 또는 평가, 교통수단·교통시설·교통체계의 주요 작동 및 운영, 공공서비스 관련 국가기관등의 의사결정, 유아·초등·중등교육에서의 학생 평가, 그리고 대통령령으로 정하는 다른 영역입니다. [S2]

참고로 인공지능사업자는 인공지능 또는 이를 이용한 제품·서비스를 제공할 때 그 인공지능이 고영향 인공지능에 해당하는지 사전에 검토해야 하며, 필요하면 과학기술정보통신부장관에게 확인을 요청할 수 있습니다. [S1]

Sources:
S1 제33조
S2 제2조
S3 제35조
S4 제34조


In [30]:
article_2_chunks = [
    chunk
    for chunk in chunks
    if chunk.get("article") == "제2조"
]

print("제2조 청크 수:", len(article_2_chunks))

for chunk in article_2_chunks:
    print(
        f"{chunk.get('chunk_role')}/{chunk.get('unit_level')}",
        chunk.get("paragraph"),
        chunk.get("item"),
        chunk.get("subitem"),
        chunk.get("title"),
    )
    print(chunk.get("content", ""))
    print()

제2조 청크 수: 23
child/subitem  4호 다목 정의
다. 「보건의료기본법」 제3조제1호에 따른 보건의료의 제공 및 이용체계의 구축ㆍ운영

child/item  3호  정의
3. "인공지능기술"이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다.

child/subitem  4호 바목 정의
바. 범죄 수사나 체포 업무를 위한 생체인식정보(얼굴ㆍ지문ㆍ홍채 및 손바닥 정맥 등 개인을 식별할 수 있는 신체적ㆍ생리적ㆍ행동적 특징에 관한 개인정보를 말한다)의 분석ㆍ활용

child/subitem  4호 가목 정의
가. 「에너지법」 제2조제1호에 따른 에너지의 공급

child/item  10호  정의
10. "인공지능사회"란 인공지능을 통하여 산업ㆍ경제, 사회ㆍ문화, 행정 등 모든 분야에서 가치를 창출하고 발전을 이끌어가는 사회를 말한다.

child/item  11호  정의
11. "인공지능윤리"란 인간의 존엄성에 대한 존중을 기초로 하여, 국민의 권익과 생명ㆍ재산을 보호할 수 있는 안전하고 신뢰할 수 있는 인공지능사회를 구현하기 위하여 인공지능의 개발, 제공 및 이용 등 모든 영역에서 사회구성원이 지켜야 할 윤리적 기준을 말한다.

child/item  1호  정의
1. "인공지능"이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다.

child/subitem  4호 차목 정의
차. 「교육기본법」 제9조제1항에 따른 유아교육ㆍ초등교육 및 중등교육에서의 학생 평가

child/subitem  4호 카목 정의
카. 그 밖에 사람의 생명ㆍ신체의 안전 및 기본권 보호에 중대한 영향을 미치는 영역으로서 대통령령으로 정하는 영역

child/subitem  4호 라목 정의
라. 「의료기기법」 제2조제1항에 따른 의료기기 및 「디지털의료제품법」 제2조제2호에 따른 디지털의료기기의 개발 및 이용

anchor/item  4호  정의
4. "고영향 인공지능"이란 사람의 

## 10. Answer 평가 기준

각 답변을 다음 기준으로 기록합니다.

| 기준 | 0점 | 1점 | 2점 |
|---|---|---|---|
| 정답성 | 핵심 내용이 틀림 | 일부만 맞거나 조건 누락 | 핵심 내용과 조건이 정확함 |
| 관련성 | 질문과 무관함 | 일부 관련되지만 우회적임 | 질문에 직접 답함 |
| 근거 충실성 | 주요 주장이 근거로 지지되지 않음 | 일부 주장만 지지됨 | 모든 핵심 주장이 근거로 지지됨 |

Hallucination은 별도로 `있음/없음`으로 기록합니다.

아래 표에는 사람이 원문과 답변을 비교해 점수를 입력합니다. LLM Judge를 사용할 수 있지만, 최종 확인은 사람이 수행합니다.


In [31]:
ANSWER_EVAL_PATH = PROJECT_ROOT / "eval" / "answer_evaluation.csv"

answer_evaluation_columns = [
    "question_id", "question", "method", "answer",
    "source_articles", "correctness_0_2", "relevance_0_2",
    "faithfulness_0_2", "hallucination",
    "evaluation_status", "notes",
]

if ANSWER_EVAL_PATH.exists():
    answer_evaluation = pd.read_csv(ANSWER_EVAL_PATH)
    for column in answer_evaluation_columns:
        if column not in answer_evaluation.columns:
            answer_evaluation[column] = pd.NA
    answer_evaluation = answer_evaluation[answer_evaluation_columns]
else:
    answer_evaluation = pd.DataFrame(columns=answer_evaluation_columns)

def record_answer_evaluation(
    question_id,
    question,
    method,
    answer_result,
    correctness=None,
    relevance=None,
    faithfulness=None,
    hallucination=None,
    notes="",
):
    global answer_evaluation

    scores = {
        "correctness": correctness,
        "relevance": relevance,
        "faithfulness": faithfulness,
    }
    for name, score in scores.items():
        if score is not None and score not in (0, 1, 2):
            raise ValueError(f"{name} 점수는 0, 1, 2 중 하나여야 합니다.")
    if hallucination not in (None, "있음", "없음"):
        raise ValueError("hallucination은 '있음' 또는 '없음'이어야 합니다.")

    evaluated = (
        all(score is not None for score in scores.values())
        and hallucination is not None
    )
    row = {
        "question_id": question_id,
        "question": question,
        "method": method,
        "answer": answer_result["answer"],
        "source_articles": ", ".join(dict.fromkeys(
            source.get("article", "")
            for source in answer_result["sources"]
            if source.get("article")
        )),
        "correctness_0_2": correctness,
        "relevance_0_2": relevance,
        "faithfulness_0_2": faithfulness,
        "hallucination": hallucination,
        "evaluation_status": "평가완료" if evaluated else "미평가",
        "notes": notes,
    }

    # 같은 질문과 방법을 다시 기록하면 중복 추가하지 않고 갱신합니다.
    duplicate = (
        (answer_evaluation["question_id"].astype(str) == str(question_id))
        & (answer_evaluation["method"].astype(str) == str(method))
    )
    answer_evaluation = answer_evaluation.loc[~duplicate]
    answer_evaluation = pd.concat(
        [answer_evaluation, pd.DataFrame([row])],
        ignore_index=True,
    )
    answer_evaluation.to_csv(
        ANSWER_EVAL_PATH, index=False, encoding="utf-8-sig"
    )
    print(f"답변 평가 기록 완료: {ANSWER_EVAL_PATH}")
    return answer_evaluation.tail(1)

# 아래 네 값을 사람이 원문과 답변을 비교하여 입력하세요.
# 아직 평가하지 않았다면 None으로 두면 답변만 '미평가' 상태로 저장됩니다.
CORRECTNESS = None       # 0, 1, 2
RELEVANCE = None         # 0, 1, 2
FAITHFULNESS = None      # 0, 1, 2
HALLUCINATION = None     # '있음' 또는 '없음'
NOTES = ""

current_golden = next(
    (item for item in golden_set if item.get("question") == query),
    None,
)
current_question_id = current_golden["id"] if current_golden else "single_query"
current_method = "Hybrid+Reranking" if USE_RERANK else "Hybrid"

recorded_answer = record_answer_evaluation(
    question_id=current_question_id,
    question=query,
    method=current_method,
    answer_result=answer_result,
    correctness=CORRECTNESS,
    relevance=RELEVANCE,
    faithfulness=FAITHFULNESS,
    hallucination=HALLUCINATION,
    notes=NOTES,
)
display(recorded_answer)
display(answer_evaluation)


답변 평가 기록 완료: /home/aiuser/rag_mini_pjt_template/eval/answer_evaluation.csv


,question_id,question,method,answer,source_articles,correctness_0_2,relevance_0_2,faithfulness_0_2,hallucination,evaluation_status,notes
0,q01,고영향 인공지능이란 무엇인가요?,Hybrid,"고영향 인공지능이란, 사람의 생명·신체의 안전 및 기본권에 중대한 영향을 미치거나 ...","제33조, 제2조, 제35조, 제34조",None,None,None,None,미평가,


,question_id,question,method,answer,source_articles,correctness_0_2,relevance_0_2,faithfulness_0_2,hallucination,evaluation_status,notes
0,q01,고영향 인공지능이란 무엇인가요?,Hybrid,"고영향 인공지능이란, 사람의 생명·신체의 안전 및 기본권에 중대한 영향을 미치거나 ...","제33조, 제2조, 제35조, 제34조",None,None,None,None,미평가,


## 11. 최종 선택 기준

- Retrieval: Hit@5와 Recall@5를 우선 비교한다.
- 성능이 비슷하면 MRR@5가 높은 방법을 선택한다.
- C와 C+D의 차이가 작으면 실행 시간이 짧은 C를 선택한다.
- D가 정답 근거의 순위를 실제로 개선할 때 C+D를 사용한다.
- Retrieval 점수가 높더라도 답변의 근거 충실성이 낮거나 Hallucination이 발생하면 최종 방법으로 채택하지 않는다.


## 실제 평가 결과 정리

## 45문항 Retrieval 평가 결과

| 방법 | Hit@5 | Recall@5 | MRR@5 |
|---|---:|---:|---:|
| Dense | 0.8095 | 0.7738 | 0.6702 |
| Hybrid(C) | **0.9286** | **0.8929** | 0.7210 |
| Hybrid+Reranking(C+D) | 0.9048 | 0.8810 | **0.8452** |

Dense보다 Hybrid가 모든 지표에서 개선됐습니다.

- Hit@5: `0.8095 → 0.9286`
- Recall@5: `0.7738 → 0.8929`
- MRR@5: `0.6702 → 0.7210`

리랭킹을 적용하면 MRR은 크게 개선됐습니다.

- MRR@5: `0.7210 → 0.8452`
- 예: q01의 정답 제2조가 2위에서 1위로 상승

하지만 일부 문항에서는 정답 조문이 밀려나면서 검색 범위가 조금 낮아졌습니다.

- Hit@5: `0.9286 → 0.9048`
- Recall@5: `0.8929 → 0.8810`
- q44: Hybrid에서는 정답 관련 조문인 제6조가 1위였지만, 리랭킹 후 2위로 하락
- q45: 모든 방법에서 관련 조문인 제21조 검색 실패

## 최종 선택

현재 평가 기준이 Hit@5와 Recall@5를 우선하는 것이므로 최종 Retrieval 방법은 `Hybrid(C)`가 적절합니다.

리랭킹은 정답 조문의 평균 순위를 높였지만, 일부 정답 근거를 Top-5 밖으로 밀어내 Hit@5와 Recall@5를 낮췄습니다. 따라서 현재 방식 그대로라면 C+D보다 C를 선택하는 것이 타당합니다.

> 최종 선택: Hybrid Search(C)  
> 보조 실험: 리랭킹은 순위 개선 효과가 있으나 검색 누락이 증가해 최종 방식에서는 제외

답변 평가는 현재 Hybrid 답변 1건만 평가됐으며, 정답성·관련성·근거 충실성 모두 2점이고 Hallucination은 없었습니다. 전체 답변 품질을 확정하려면 더 많은 문항의 답변 평가가 필요합니다.

참고로 q41~q45는 범위 밖 질문입니다. 이런 문항은 Retrieval Hit보다 시스템이 “법률에서 확인할 수 없다”고 적절히 거절하는지를 별도 `Abstention Accuracy`로 평가하는 것이 더 정확합니다.